In [2]:
"""
Load a SALM YAML config and read/print one batch from the dataset.

Pipeline:
    YAML (manifest_filepath / cuts_path / input_cfg)
        -> get_lhotse_dataloader_from_config(...)   # reads manifest -> CutSet
        -> SALMDataset.__getitem__(cuts)            # collates audio + tokens
"""

import argparse
from pprint import pprint

import torch
from omegaconf import OmegaConf

from nemo.collections.common.data.lhotse import get_lhotse_dataloader_from_config
from nemo.collections.common.tokenizers import AutoTokenizer
from nemo.collections.speechlm2.data import SALMDataset


def _ensure_single_token(tokenizer, token: str) -> int:
    """
    Make sure `token` exists in the tokenizer vocab and maps to exactly ONE id.

    SALMDataset requires this for the audio_locator_tag. NeMo's AutoTokenizer
    wrapper forwards the dict straight to HF, which (since transformers v5)
    accepts only a dict — never a list — so we always use the dict form.
    """
    # Already a single token? done.
    ids = tokenizer.text_to_ids(token)
    if len(ids) == 1:
        return ids[0]

    payload = {"additional_special_tokens": [token]}
    added = False

    # Path 1: NeMo AutoTokenizer wrapper (forwards to HF underneath).
    if hasattr(tokenizer, "add_special_tokens"):
        try:
            tokenizer.add_special_tokens(payload)
            added = True
        except Exception as e:
            print(f"[warn] tokenizer.add_special_tokens failed: {e!r} — trying HF fallback")

    # Path 2: reach down to the underlying HF tokenizer directly.
    if not added and hasattr(tokenizer, "tokenizer"):
        hf_tok = tokenizer.tokenizer
        if hasattr(hf_tok, "add_special_tokens"):
            hf_tok.add_special_tokens(payload)
            added = True

    if not added:
        raise RuntimeError(
            f"Could not register special token {token!r} on tokenizer of type "
            f"{type(tokenizer).__name__}."
        )

    ids = tokenizer.text_to_ids(token)
    if len(ids) != 1:
        raise RuntimeError(
            f"After registration, {token!r} still maps to {ids} (expected 1 id). "
            "Tokenizer may need a fast/slow refresh."
        )
    return ids[0]


def main(config_path: str, split: str = "train_ds", num_batches: int = 1):
    # 1. Load YAML config (e.g. examples/speechlm2/conf/salm.yaml)
    cfg = OmegaConf.load(config_path)

    # 2. Build the tokenizer the SAME way SALM does it. Most SALM configs
    #    point at a HF LLM via `model.pretrained_llm`. Adjust if your config
    #    stores the tokenizer somewhere else.
    pretrained_llm = cfg.model.pretrained_llm
    tokenizer = AutoTokenizer(pretrained_model_name=pretrained_llm)

    # 2b. Register the audio-locator tag as a single special token. The SALM
    #     model normally does this inside its __init__; when we build the
    #     dataset standalone we must do it ourselves, otherwise the tag gets
    #     BPE-split into many pieces and SALMDataset will raise:
    #         ValueError: Special token '<|audioplaceholder|>' should map to
    #         exactly one id, got [...].
    audio_locator_tag = cfg.model.get("audio_locator_tag", "<|audioplaceholder|>")
    _ensure_single_token(tokenizer, audio_locator_tag)

    # 3. Instantiate the dataset. SALMDataset only collates; manifest reading
    #    happens inside get_lhotse_dataloader_from_config below.
    pattern = cfg.data.get("pattern", "continuation")  # "continuation" | "repetition" | "mixed"
    dataset = SALMDataset(
        tokenizer=tokenizer,
        audio_locator_tag=audio_locator_tag,
        pattern=pattern,
    )

    # 4. Pick the split section from the YAML (train_ds / validation_ds / test_ds).
    #    First resolve interpolations like ${model.prompt_format} against the FULL
    #    config — once we slice out cfg.data.train_ds the parent context is lost
    #    and OmegaConf raises InterpolationKeyError.
    resolved = OmegaConf.to_container(cfg, resolve=True)
    ds_cfg = OmegaConf.create(resolved["data"][split])

    # validation/test sections are usually nested under `datasets:` — flatten the first one.
    if "datasets" in ds_cfg:
        first_name, first_cfg = next(iter(ds_cfg.datasets.items()))
        print(f"[info] using validation/test sub-dataset: {first_name}")
        merged = OmegaConf.merge(
            {k: v for k, v in ds_cfg.items() if k != "datasets"},
            first_cfg,
        )
        ds_cfg = merged

    # 5. Build the Lhotse dataloader. THIS is where the manifest is actually read.
    dataloader = get_lhotse_dataloader_from_config(
        config=ds_cfg,
        global_rank=0,
        world_size=1,
        dataset=dataset,
        tokenizer=tokenizer,
    )

    # 6. Pull a few batches and print them.
    for i, batch in enumerate(dataloader):
        if batch is None:
            print(f"[batch {i}] all cuts filtered out (audio load failures)")
            continue

        print(f"\n========== batch {i} ==========")
        print(f"audios:     {tuple(batch['audios'].shape)}  dtype={batch['audios'].dtype}")
        print(f"audio_lens: {batch['audio_lens'].tolist()}")
        print(f"input_ids:  {tuple(batch['input_ids'].shape)}")
        print(f"loss_mask:  {tuple(batch['loss_mask'].shape)}  "
              f"target_tokens={int(batch['loss_mask'].sum())}")

        # Decode the first sequence in the batch so you can eyeball what the
        # model actually sees (audio_locator_tag included).
        decoded = tokenizer.ids_to_text(batch["input_ids"][0].tolist())
        print(f"\nfirst example decoded:\n{decoded!r}")

        # Show the underlying conversation object (turns, roles, audio paths).
        print("\nfirst conversation turns:")
        pprint(batch["conversations"][0].turns)

        if i + 1 >= num_batches:
            break


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-04-27 08:20:47 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
      warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
    


[NeMo I 2026-04-27 08:20:53 ear_tts_model:111] Triton available & CUDA detected. Using Triton kernel for batch_matmul.


In [12]:
main('/home/salm/conf/salm_gemma4_v2_1.yaml', split="train_ds", num_batches=1)

[NeMo I 2026-04-27 08:15:07 auto_tokenizer:252] 1 special tokens added, resize your model accordingly.
[NeMo I 2026-04-27 08:15:07 dataloader:342] We will be using a Lhotse DataLoader.


[NeMo W 2026-04-27 08:15:07 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


[NeMo I 2026-04-27 08:15:07 dataloader:659] Creating a Lhotse DynamicCutSampler (bucketing is disabled, (max_batch_duration=None max_batch_size=4)

========== batch 0 ==========
audios:     (4, 288379)  dtype=torch.float32
audio_lens: [56481, 288379, 90453, 90880]
input_ids:  (4, 44)
loss_mask:  (4, 44)  target_tokens=69

first example decoded:
'Write down what is said in this recording:Tennis courts have been built for youngsters to learn the sport.'

first conversation turns:
[TextTurn(value='Write down what is said in this recording:', role='user'),
 AudioTurn(cut=MonoCut(id='001332',
                       start=0.0,
                       duration=3.5300625,
                       channel=0,
                       supervisions=[SupervisionSegment(id='001332',
                                                        recording_id='001332',
                                                        start=0,
                                                        duration=3.5300625,
     

In [3]:
main('/home/salm/conf/salm_gemma4_v2_2.yaml', split="train_ds", num_batches=1)

[NeMo I 2026-04-27 08:20:56 auto_tokenizer:252] 1 special tokens added, resize your model accordingly.
[NeMo I 2026-04-27 08:20:56 dataloader:342] We will be using a Lhotse DataLoader.
[NeMo I 2026-04-27 08:20:56 dataloader:659] Creating a Lhotse DynamicCutSampler (bucketing is disabled, (max_batch_duration=None max_batch_size=4)

========== batch 0 ==========
audios:     (4, 155520)  dtype=torch.float32
audio_lens: [155520, 71202, 17185, 40001]
input_ids:  (4, 22)
loss_mask:  (4, 22)  target_tokens=35

first example decoded:
'Transcribe the following:Not only football, people also get enjoyed in different kinds of sports where they forget'

first conversation turns:
[TextTurn(value='Transcribe the following:', role='user'),
 AudioTurn(cut=MonoCut(id='_home_commotion_asr_dataset_svarah_train_en_audio_001731.wav',
                       start=0.0,
                       duration=9.72,
                       channel=0,
                       supervisions=[SupervisionSegment(id='_home_com

In [4]:
main('/home/salm/conf/salm_gemma4_v3.yaml', split="train_ds", num_batches=1)

[NeMo I 2026-04-27 08:21:27 auto_tokenizer:252] 1 special tokens added, resize your model accordingly.
[NeMo I 2026-04-27 08:21:27 dataloader:342] We will be using a Lhotse DataLoader.


[NeMo W 2026-04-27 08:21:27 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: dataset_pattern


[NeMo I 2026-04-27 08:21:27 dataloader:659] Creating a Lhotse DynamicCutSampler (bucketing is disabled, (max_batch_duration=None max_batch_size=4)

========== batch 0 ==========
audios:     (22, 273065)  dtype=torch.float32
audio_lens: [78683, 106563, 30198, 70403, 114040, 128662, 6598, 53865, 78099, 151046, 273065, 65840, 14460, 43165, 114560, 8337, 36404, 46080, 75982, 15121, 48290, 70401]
input_ids:  (4, 76)
loss_mask:  (4, 76)  target_tokens=220

first example decoded:
'Transcribe the following:United States, Australia, Canada, Germany, Chinathere are Vatsavitri actually, Vatsavitri the married women they celebratedelete last item listedWhy are there no discount on Seafoods, when other categories have it?So while running they should have a common line of landing.'

first conversation turns:
[TextTurn(value='Transcribe the following:', role='user'),
 AudioTurn(cut=MonoCut(id='006183',
                       start=0.0,
                       duration=4.9176875,
                      